In [1]:
!pip install tensorflow nltk pandas numpy scikit-learn

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Download NLTK stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Load dataset (use only 100,000 rows)
data = pd.read_csv('/content/training.1600000.processed.noemoticon.csv', encoding='latin-1', names=['sentiment', 'id', 'date', 'query', 'user', 'text'], nrows=100000)
data = data[['text', 'sentiment']]
data['sentiment'] = data['sentiment'].replace({0: 0, 4: 1})  # 0: negative, 1: positive

# Clean text
def preprocess_text(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)  # Remove URLs
    text = re.sub(r'@\w+|\#', '', text)  # Remove mentions and hashtags
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = text.lower()  # Convert to lowercase
    text = ' '.join([word for word in text.split() if word not in stop_words])  # Remove stopwords
    return text

data['text'] = data['text'].apply(preprocess_text)

# Convert text to numbers
max_words = 3000  # Smaller vocabulary
max_len = 50      # Shorter sequence length
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(data['text'])
sequences = tokenizer.texts_to_sequences(data['text'])
padded_sequences = pad_sequences(sequences, maxlen=max_len)

# Prepare labels
labels = np.array(data['sentiment'])

# Split data
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, labels, test_size=0.2, random_state=42)

# Build LSTM model
model = Sequential()
model.add(Embedding(max_words, 64, input_length=max_len))  # Smaller embedding
model.add(LSTM(32, return_sequences=False))               # Smaller LSTM
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

# Compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train model
history = model.fit(X_train, y_train, epochs=3, batch_size=32, validation_split=0.2)

# Evaluate model
y_pred = (model.predict(X_test) > 0.5).astype(int)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

# Predict on new text
def predict_sentiment(text):
    text = preprocess_text(text)
    sequence = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(sequence, maxlen=max_len)
    prediction = model.predict(padded)[0][0]
    return "Positive" if prediction > 0.5 else "Negative"

# Test with a sample tweet
sample_text = "I love this product, it's amazing!"
print("Sentiment:", predict_sentiment(sample_text))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/3
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.9978 - loss: 0.0332 - val_accuracy: 1.0000 - val_loss: 1.2496e-05
Epoch 2/3
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - accuracy: 1.0000 - loss: 8.2368e-05 - val_accuracy: 1.0000 - val_loss: 1.7080e-06
Epoch 3/3
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - accuracy: 1.0000 - loss: 2.5095e-05 - val_accuracy: 1.0000 - val_loss: 3.1410e-07
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
Accuracy: 1.0
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     20000

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step
Sentiment: Negative


In [3]:
print(predict_sentiment("This movie is terrible!"))
print(predict_sentiment("Wow, what a great day!"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Negative
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
Negative
